# 1. Mount google drive

In [51]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
# Set working directory

import os, sys, shutil
repo_root = "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio"
colab_dir = os.path.join(repo_root, "colab")

# Ensure eplus utility is copied for imports
eplus_src = os.path.join(repo_root, "EnergyPlus utility")
eplus_dest = os.path.join(colab_dir, "eplus")
if not os.path.exists(eplus_dest):
    shutil.copytree(eplus_src, eplus_dest)

os.chdir(colab_dir)
sys.path.append(colab_dir)
print(f"Working directory set to: {os.getcwd()}")


Working directory set to: /content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab


# 2. Ollama local run

In [53]:
# 1. Install missing dependencies & GPU utils
!sudo apt-get install -y zstd
!sudo apt update && sudo apt install pciutils lshw -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 108 not upgraded.
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading packag

In [54]:
# 2. Install Ollama Linux Engine
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [55]:
# 3. Install Python SDK
!pip install ollama

In [5]:
# 1. Kill any existing instances
!pkill ollama

In [20]:
# 4. Set host environment variables to allow connections
import os
import subprocess
import time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

In [21]:
# Tell Ollama to save weights to your persistent Google Drive folder:
os.environ["OLLAMA_MODELS"] = "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab/ollama_models"


In [56]:
# 2. Safely copy the massive brain from Drive to Colab's fast internal disk (~30 seconds)
print("Copying model from Drive to local disk...")
!mkdir -p /root/.ollama/models
!cp -r "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab/ollama_models/"* /root/.ollama/models/
print("Copy complete!")

Copying model from Drive to local disk...
Copy complete!


In [88]:
# 3. Boot Server normally (it will use the fast internal disk, NOT Drive)
!nohup ollama serve > ollama.log 2>&1 &
time.sleep(5)
print("✅ Ollama Server Awake and ready!")

✅ Ollama Server Awake and ready!


In [89]:
# 6. Test if it is running
!curl http://localhost:11434

Ollama is running

In [ ]:
# 7. Pull the model!
!ollama pull gemma3:4b

In [ ]:
!ollama rm gemma3:270m

In [82]:
!ollama list

NAME         ID              SIZE      MODIFIED    
gemma3:4b    a2af6cc3eb7f    3.3 GB    3 hours ago    


In [83]:
"""### Testing on Text Input"""

import ollama
response = ollama.chat(model='gemma3:4b', messages=[
  {'role': 'user', 'content': 'what is rgb, answer short'},
])
print(response['message']['content'])

RGB stands for Red, Green, and Blue. It’s a color model used to create colors by combining these three primary colors in various proportions. 🎨💻



# SmartHVAC Studio: Backend Worker (Layer 3)
This executable notebook acts as the **Coordination Engine**. It polls Firebase for new jobs, runs AI generation (Layer 4), executes EnergyPlus simulations (Layer 5), and uploads results.

---

## 1. Mount drive

In [71]:
import os, sys, importlib

# 1. Forcefully remove old ghost folders so your Drive is clean
if os.path.exists('../sim_runs'):
    !rm -rf ../sim_runs/
if os.path.exists('sim_runs'):
    !rm -rf sim_runs/
if os.path.exists('RunFiles'):
    !rm -rf RunFiles/

# 2. Forcefully fix Base.idf directly on the server to strip the Duplicate Zone
base_path = 'templates/Base.idf'
if os.path.exists(base_path):
    with open(base_path, 'r') as f:
        lines = f.readlines()

    # Strip everything from the first "Zone," onwards to remove duplicated geometry
    with open(base_path, 'w') as f:
        for i, line in enumerate(lines):
            # If we hit the word "Zone," we stop writing the rest of the template
            if "Zone," in line.strip() and "Zone ONE" not in line and "Zone," == line.strip():
                pass # Just a safety
            if line.strip() == "Zone,":
                break
            f.write(line)
    print("Fixed Base.idf successfully!")
else:
    print(f"Could not find {base_path} - ensure you are in the correct directory.")

# 3. Force Python to reload the fixed python files from disk
sys.path.append(os.getcwd())
import backend.geometry_util as geometry_util
import simulation_runner
importlib.reload(geometry_util)
importlib.reload(simulation_runner)

print("✅ Server Cache Cleared, Cleaned Drive, and Hard-Patched the IDF File! You are ready to simulate.")


Fixed Base.idf successfully!
✅ Server Cache Cleared, Cleaned Drive, and Hard-Patched the IDF File! You are ready to simulate.


In [ ]:
!python verify_phase2.py

## 2. Setup Environment

In [63]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies installed.


In [72]:
import sys
import os

# Add the local folder to path so we can import 'backend'
sys.path.append(os.getcwd())

print("Current Working Directory:", os.getcwd())
print("Python Path Updated.")

Current Working Directory: /content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab
Python Path Updated.


## 3. Authentication

In [73]:
# Check for Service Account Key
key_path = "serviceAccountKey.json"

if not os.path.exists(key_path):
    print("❌ ERROR: serviceAccountKey.json not found.")
    print("Please upload your Firebase Service Account JSON key to the file browser on the left.")
else:
    print("✅ Found serviceAccountKey.json")

✅ Found serviceAccountKey.json


## 3. Initialize Modules

In [74]:
import firebase_admin
from firebase_admin import credentials, firestore, storage
import json
import time
import difflib # For Diff Viewer

In [75]:
from backend.firebase_connector import FirebaseConnector
from backend.ai_generator import AIPipelines
from simulation_runner import run_simulation_job
import time

# Initialize Connections
try:
    fb = FirebaseConnector(key_path)
    ai = AIPipelines() # API Key can be passed here or set in ENV
    print("Backend Modules Initialized Successfully.")
except Exception as e:
    print("Initialization Failed:", e)

[Firebase] Connected to project: smarthvac-studio-b84c4
Backend Modules Initialized Successfully.


## 4. Verify epw found and downloaded to runtime

In [76]:
import os

# Expected path relative to Colab root
file_path = "weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw"

if os.path.exists(file_path):
    print(f"✅ FOUND: {file_path}")
    print(f"   Size: {os.path.getsize(file_path)/1024:.1f} KB")
else:
    print(f"❌ MISSING: {file_path}")
    if os.path.exists("weather"):
        print(f"   Contents of 'weather/': {os.listdir('weather')}")
    else:
        print("   'weather' folder does not exist!")


✅ FOUND: weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw
   Size: 1610.1 KB


## 5. Verify Base.idf found and downloaded to runtime

In [77]:
import os
import firebase_admin
from firebase_admin import storage

def download_template(template_name="Base.idf"):
    """Downloads the IDF template from Firebase Storage to local Colab."""
    try:
        bucket = storage.bucket()
        blob_path = f"templates/{template_name}"
        local_path = f"templates/{template_name}"

        # Ensure local dir exists
        os.makedirs("templates", exist_ok=True)

        blob = bucket.blob(blob_path)
        if blob.exists():
            blob.download_to_filename(local_path)
            print(f"✅ Downloaded Template: {local_path}")
            return True
        else:
            print(f"⚠️ Template not found in Storage: {blob_path}")
            print("   (Please create 'templates' folder in Firebase Storage and upload Base.idf)")
            return False

    except Exception as e:
        print(f"❌ Failed to download template: {e}")
        return False

        return False

def generate_and_upload_diff(base_path, new_path, job_id):
    """Generates an HTML diff between Base.idf and in.idf and uploads it."""
    try:
        # Read files
        with open(base_path, 'r') as f:
            base_lines = f.readlines()
        with open(new_path, 'r') as f:
            new_lines = f.readlines()

        # Generate HTML Diff using Python's built-in difflib
        diff_html = difflib.HtmlDiff().make_file(
            base_lines, new_lines,
            fromdesc='Base Template',
            todesc='Generated IDF',
            context=True,  # Show only changes + context
            numlines=3     # 3 lines of context
        )

        # Save locally
        diff_path = f"jobs/{job_id}/diff.html"
        with open(diff_path, "w") as f:
            f.write(diff_html)

        # Upload to Storage (Content-Type is CRITICAL for browser viewing)
        bucket = storage.bucket()
        blob = bucket.blob(f"jobs/{job_id}/diff.html")
        blob.upload_from_filename(diff_path, content_type="text/html")
        print(f"   📊 Uploaded Diff HTML to Storage.")
        return True
    except Exception as e:
        print(f"   ❌ Failed to generate diff: {e}")
        return False

# Call this on startup
download_template()


✅ Downloaded Template: templates/Base.idf


True

## 6. Main polling loop

In [ ]:
import time
from datetime import datetime
import traceback

def run_backend_loop(db):
    print("🚀 Backend is running... Waiting for jobs.")
    print("   - Watching 'jobs' collection for 'queued' tasks.")
    print("   - Watching 'test_connectivity' collection for 'test_connection' tasks.")

    while True:
        try:
            # ---------------------------------------------------------
            # 1. Check for CONNECTION TESTS (Fast Path)
            # ---------------------------------------------------------
            test_docs = db.collection("test_connectivity").where("status", "==", "test_connection").stream()

            for doc in test_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔹 Found Connection Test: {job_id}")

                # Get Toggles (Default to True if missing)
                do_gemini = data.get('checkGemini', True)
                do_openai = data.get('checkOpenAI', True)
                do_hf = data.get('checkHF', True)

                # Perform checks using your existing AI class
                # Pass the flags to the method
                results = ai.test_connections(check_openai=do_openai, check_gemini=do_gemini, check_hf=do_hf)

                # Update Firestore
                db.collection("test_connectivity").document(job_id).update({
                    "status": "tested",
                    "testResults": results,
                    "processedAt": datetime.now()
                })
                print(f"   ✅ Test Completed. Updated {job_id}")

            # ---------------------------------------------------------
            # 2. Check for SIMULATION JOBS (Normal Path)
            # ---------------------------------------------------------
            job_docs = db.collection("jobs").where("status", "==", "queued").stream()

            for doc in job_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔸 Found Queued Job: {job_id}")

                # Mark as running immediately
                db.collection("jobs").document(job_id).update({"status": "running"})

                try:
                    # Extract Inputs from new UI
                    nlp_input = data.get("nlpInputText", "")
                    sim_settings = data.get("simulationConfig", {})

                    # Log what we received
                    print(f"   Input: {nlp_input[:50]}...")
                    print(f"   Settings: {sim_settings}")


                    # --- PHASE 3: THE SYSTEM LOOP ---
                    run_mode = data.get("runMode", "ai")

                    import os
                    idf_save_path = f"RunFiles/{job_id}_in.idf"
                    os.makedirs("RunFiles", exist_ok=True)

                    if run_mode == "minimal":
                        print("   [Phase 3] Safe Test Mode: Using Minimal.idf bypass...")
                        import shutil
                        # minimal_path = "../Example IDF files/Minimal.idf"
                        minimal_path = "../Example IDF files/Exercise1A.idf"

                        if not os.path.exists(minimal_path):
                            raise Exception(f"Could not find Minimal.idf at {minimal_path}")
                        shutil.copy(minimal_path, idf_save_path)
                    else:
                        selected_model = data.get("selectedModel", "openai")
                        print(f"   [Phase 3] Generating IDF using {selected_model}...")

                        final_idf_string = ai.generate_idf_from_text(nlp_input, sim_settings, model_type=selected_model)

                        if final_idf_string.startswith("! Error"):
                            raise Exception(f"AI Generation Failed: {final_idf_string}")

                        # Save IDF
                        with open(idf_save_path, "w", encoding="utf-8") as f:
                            f.write(final_idf_string)


                    # Epw path usually defaults to "weather.epw" or from sim_settings
                    epw_name = sim_settings.get("weather_file", "weather.epw")
                    if not os.path.exists(epw_name) and os.path.exists("weather/" + epw_name):
                         epw_name = "weather/" + epw_name
                    elif not os.path.exists(epw_name):
                         epw_name = "weather.epw"

                    sim_results = simulation_runner.run_simulation_job(
                        job_id=job_id,
                        idf_path=idf_save_path,
                        epw_path=epw_name,
                        config=sim_settings,
                        output_dir_base="sim_runs"
                    )
                    print("   [Phase 3] Simulation Complete! Uploading results to Storage...")

                    # Upload to Storage
                    bucket = storage.bucket()
                    result_urls = {}

                    for key, file_path in sim_results.items():
                        if os.path.exists(file_path):
                            blob_path = f"jobs/{job_id}/{os.path.basename(file_path)}"
                            blob = bucket.blob(blob_path)

                            # c_type = "image/png" if key == "plot" else "text/csv"
                            #c_type = "image/png" if key.startswith("plot") else "text/csv"

                            if key.startswith("plot"):
                                c_type = "image/png"
                            elif key == "idf":
                                c_type = "text/plain"
                            elif key == "summary":            # <--- ADD THIS LINE
                                c_type = "text/html"          # <--- ADD THIS LINE
                            else:
                                c_type = "text/csv"


                            blob.upload_from_filename(file_path, content_type=c_type)

                            # Store the direct URL
                            try:
                                blob.make_public()
                                result_urls[key] = blob.public_url
                            except:
                                result_urls[key] = f"gs://{bucket.name}/{blob_path}"

                    db.collection("jobs").document(job_id).update({
                        "status": "done",
                        "resultPaths": result_urls,
                        "completedAt": datetime.now()
                    })
                    print(f"   ✅ Job Finished: {job_id}\n")

                except Exception as e:
                    error_msg = str(e)
                    print(f"   ❌ Job Failed: {error_msg}")
                    traceback.print_exc()

                    db.collection("jobs").document(job_id).update({
                        "status": "error",
                        "errorMessage": error_msg
                    })

            # Sleep to prevent quota exhaustion
            time.sleep(3)

        except Exception as main_e:
            print(f"⚠️ Loop Error: {main_e}")
            time.sleep(5)

# ---------------------------------------------------------------------
# START THE LOOP (Runs indefinitely)
# Ensure 'fb' is defined (from previous cells)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    if 'fb' in globals():
        # Access the internal firestore client from your connector
        if hasattr(fb, 'db'):
            db_client = fb.db
        else:
            # Fallback if fb is the client itself
            db_client = fb

        run_backend_loop(db_client)
    else:
        print("❌ Error: 'fb' variable not found. Please run the Firebase Initialization cell first.")


🚀 Backend is running... Waiting for jobs.
   - Watching 'jobs' collection for 'queued' tasks.
   - Watching 'test_connectivity' collection for 'test_connection' tasks.
🔸 Found Queued Job: job_20260428_232545
   Input: Create a 15x15 building. I want the exterior walls...
   Settings: {'weather_file': 'USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw', 'run_type': 'design_day'}
   [Phase 3] Generating IDF using ollama...
[AI] Generating Modular IDF using model: ollama
[AI RAG] Running Pass 1 (Planner) to extract keywords...
[AI RAG] Extracted Keywords: ['Heavy Brick', 'wood', 'insulation', 'standard', 'roof', 'brick', 'wall', 'exterior', 'building', '15x15', 'wood stud', 'insulation', 'thermal performance', 'construction', 'material', 'weather', 'climate']
[AI RAG] Filtered Menu down to 15 items: ['Light Exterior Wall', 'Medium Exterior Wall', 'Heavy Exterior Wall', 'Composite 2x4 Wood Stud R11', 'Composite 2x6 Wood Stud R19', 'Light Roof/Ceiling', 'Medium Roof/Ceiling', 'Heavy Roof/Ceiling

/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


🔸 Found Queued Job: job_20260428_232743
   Input: Simulate a U-shaped building that is 99.73 meters ...
   Settings: {'weather_file': 'USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw', 'run_type': 'design_day'}
   [Phase 3] Generating IDF using ollama...
[AI] Generating Modular IDF using model: ollama
[AI RAG] Running Pass 1 (Planner) to extract keywords...
[AI RAG] Extracted Keywords: ['U-shaped building', 'gable roof', 'wood', 'insulation', 'metal roof', 'concrete floor', 'window', 'window-to-wall ratio', 'window U-factor', 'SHGC', 'wall', 'roof', 'RSI', 'infiltration', 'ventilation', 'occupancy', 'heating setpoint', 'cooling setpoint', 'thermal zones', 'carpet', 'electric equipment density', 'light density', 'people density']
[AI RAG] Filtered Menu down to 15 items: ['Light Exterior Wall', 'Light Roof/Ceiling', 'Medium Exterior Wall', 'Medium Roof/Ceiling', 'Heavy Exterior Wall', 'Heavy Roof/Ceiling', 'Composite 2x4 Wood Stud R11', 'Composite 2x6 Wood Stud R19', 'Composite Insulated Co

/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
